# Synthetic Data Generation — Hands-On

## 0. Setup

In [ ]:
%pip install -q numpy
import numpy as np, hashlib, re
rng=np.random.RandomState(44)
rows=[{'prompt':'Summarize policy','answer':'Short summary','score':.9,'license':'ok'},{'prompt':' summarize policy ','answer':'Short summary','score':.85,'license':'ok'},{'prompt':'','answer':'bad','score':.1,'license':'ok'},{'prompt':'Legal advice','answer':'Maybe','score':.7,'license':'blocked'}]

## 1. Normalize and deduplicate

In [ ]:
def norm(s): return re.sub(r'\s+',' ',s.strip().lower())
seen=set(); kept=[]
for r in rows:
    key=(norm(r['prompt']), norm(r['answer']))
    if r['score']>=.7 and r['license']=='ok' and key[0] and key not in seen:
        seen.add(key); kept.append(r)
print(kept)

## 2. Deterministic split

In [ ]:
order=sorted(kept,key=lambda r: hashlib.sha1(norm(r['prompt']).encode()).hexdigest())
train=order[:1]; val=order[1:]; print(train,val)

## 3. Contamination check

In [ ]:
train_prompts={norm(r['prompt']) for r in train}; eval_prompts={'summarize policy','unseen eval'}
print('leaks', train_prompts & eval_prompts)

## 4. Diversity accounting

In [ ]:
tasks=np.array(['summary','qa','summary','extract','qa']); unique,counts=np.unique(tasks,return_counts=True); print(dict(zip(unique,counts)))

## 5. Quality threshold curve

In [ ]:
scores=np.array([.95,.8,.7,.55,.4]); utility=scores-.05*np.arange(len(scores));
for th in [.5,.7,.9]: print(th, int((scores>=th).sum()), round(float(utility[scores>=th].mean()),3))

## 6. Synthetic marginal gain

In [ ]:
real=np.array([100,200,400]); synth=np.array([0,200,800]); acc=.7+.08*(1-np.exp(-real/200))+.05*(1-np.exp(-synth/300)); print(np.round(acc,3))

## Exercises
Add near-duplicate similarity, license fields, and per-task sampling caps.